# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Date Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets and their fields by `@id`.

All references to record sets, fields, and columns use their unique `@id` as defined in the Croissant schema.

In [ ]:
from pprint import pprint

# List all record sets with their @id and fields (all referencing by their @id)

record_sets = metadata.recordSet

if not record_sets:
    print("No record sets found in metadata. Please check schema for available records.")
else:
    print("Available Record Sets (@id):\n------------------------------")
    for recset in record_sets:
        recset_id = recset['@id'] if isinstance(recset, dict) and '@id' in recset else recset
        print(f"- RecordSet @id: {recset_id}")
        if hasattr(metadata, 'field') and metadata.field:
            # Fields belonging to this recordset
            rs_fields = [f for f in metadata.field if f.get('cr:recordSet', '') == recset_id]
            if len(rs_fields):
                print("  Fields (@id): ")
                for field in rs_fields:
                    print(f"   - {field['@id']} (name: {field.get('schema:name','')})")
        print()

# If record sets are not present in top-level, check in distributions
if not record_sets and hasattr(metadata, 'distribution'):
    print("Checking distributions for possible record sets and columns: \n-----------------")
    for dist in metadata.distribution:
        print(f"Distribution @id: {dist['@id']}  name: {dist.get('schema:name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field or column `@id`s from the overview above.

**Note:** We reference record set(s) by their `@id`. If the schema does not expose `recordSet`, we use the distribution's `@id` (data file) directly.

In [ ]:
# Find available record_set or, if missing, use distribution

record_set_ids = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for recset in metadata.recordSet:
        recset_id = recset['@id'] if isinstance(recset, dict) and '@id' in recset else recset
        record_set_ids.append(recset_id)
elif hasattr(metadata, 'distribution') and metadata.distribution:
    for dist in metadata.distribution:
        dist_id = dist['@id'] if isinstance(dist, dict) and '@id' in dist else dist
        record_set_ids.append(dist_id)
else:
    print("No record sets or distributions found in the metadata.")

print(f"Record sets to extract: {record_set_ids}")

# Extract data from each record set (or distribution)
dataframes = {}
for recset_id in record_set_ids:
    print(f"Loading records for {recset_id} ...")
    df = pd.DataFrame(list(dataset.records(record_set=recset_id)))
    print(f"Loaded {len(df)} records with columns: {df.columns.tolist()[:6]} ...")
    dataframes[recset_id] = df

# For further analysis, choose the first available record set
chosen_recset_id = record_set_ids[0] if record_set_ids else None
print(f"\nExample Data Columns for {chosen_recset_id}:\n", dataframes[chosen_recset_id].columns.tolist()[:15])
dataframes[chosen_recset_id].head(3)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All fields are referenced by their `@id` as defined in the dataset.

In [ ]:
df = dataframes[chosen_recset_id]

# Identify a few numeric fields by their likely names or @ids
numeric_candidate_ids = [col for col in df.columns if (
    'abundance' in col.lower() or
    'coverage' in col.lower() or
    'count' in col.lower() or
    'mw' in col.lower() or
    df[col].dtype in ['int64','float64']
)]
print(f"Candidate numeric fields for EDA: {numeric_candidate_ids}")

# Choose a numeric field for filtering/normalizing
numeric_field_id = numeric_candidate_ids[0] if numeric_candidate_ids else df.columns[0]
print(f"Chosen numeric field (for demo): {numeric_field_id}")

# (Example) Filter records with value above threshold
filtered_df = df.copy()
try:
    threshold = filtered_df[numeric_field_id].quantile(0.75) # Use 75th percentile as threshold
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top quartile): {len(filtered_df)} rows")
except Exception as e:
    print(f"Could not filter on {numeric_field_id}: {e}")

# Normalize
try:
    filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head(3))
except Exception as e:
    print(f"Could not normalize {numeric_field_id}: {e}")

# Try grouping by potential categorical field (e.g., 'description', 'accession', etc. by @id)
group_field_candidates = [col for col in df.columns if (
    'accession' in col.lower() or 'description' in col.lower() or 'sample' in col.lower())
]
group_field_id = group_field_candidates[0] if group_field_candidates else None
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id}:\n", grouped_df.head())
else:
    print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields from the dataset using their `@id` as references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If grouping field exists, visualize group means
if group_field_id is not None and not filtered_df.empty:
    top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(10)
    plt.figure(figsize=(10,5))
    sns.barplot(data=top_groups, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.title(f'Mean {numeric_field_id} by {group_field_id} (Top 10)')
    plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded and explored the FAIR² mass spectrometry data using the Croissant schema and `mlcroissant`.
- Identified available record sets and data fields by their unique `@id`.
- Illustrated how to filter, normalize, and group data based on specific field `@id`s.
- Visualized numeric field distributions and group aggregates using Matplotlib and Seaborn.

This approach can be extended to more advanced analyses and supports reproducible, FAIR-aligned data science workflows.